# 03 · Factor usage across the fibroblast populations, scRNA-seq and multiome

How each of the 16 programmes is used by each fibroblast population, **separately in the two
modalities** the model was fitted on. Populations are the current harmonised annotation
(`taa_l3_harmonized`, `thyAgeing_Fb_scell_multiome_obs_harmonized_2026-09-06.csv`): one label
vocabulary — Fb-interlo, Fb-perilo, Fb-med, Fb-interm, Fb-TGFB+, Fb-adipo — in both arms, so the
cells and nuclei panels can be compared column by column and the cross-modality agreement of each
programme is a direct correlation rather than a matching by name.

**Usage is a composition.** Each cell's 16 raw usages are divided by their sum, so a value is the
share of the cell spent on that programme and cells are comparable across arms. The main heatmap
z-scores each factor **across populations within its arm**: that removes the baseline every assay
puts on a factor (S8 reaches a 0.56 mean share in perilobular nuclei while mature adipose peaks
near 0.1) and asks the comparable question — *which populations use this programme most*. The
absolute shares are kept as the second figure and as tables.

In [1]:
import os, sys, warnings
warnings.filterwarnings('ignore')
import hdf5plugin
import numpy as np
import pandas as pd
import anndata as ad
import matplotlib.pyplot as plt
from scipy.stats import spearmanr

sys.path.insert(0, '/nfs/team205/vk8/projects/thymus_ageing_atlas/FB_Vascular_compartment/notebooks/revisions/final')
import fb_common as C

C.set_style()
RES, PLT = C.out_dirs('03_usage_by_population')
pd.set_option('display.width', 220); pd.set_option('display.max_columns', 40)

light = ad.read_h5ad(C.light_path())
H = pd.read_parquet(f'{C.INMF_DIR}/H_{C.TAG}.parquet')
H = H.loc[H.index.intersection(light.obs_names)]
light = light[H.index].copy()
obs = light.obs.copy()
U = C.usage_shares(H)
MIN_CELLS = 20                                             # a population needs this many cells in an arm to be shown

# the harmonised annotation is read from its CSV, so an update there propagates without re-running 01
harm = pd.read_csv(C.HARMONIZED_OBS, index_col=0, usecols=lambda c: c in ('', 'Unnamed: 0', C.HARMONIZED_LABEL))
obs['population'] = harm[C.HARMONIZED_LABEL].reindex(obs.index).astype(str).replace({'nan': np.nan}).values
print('population x arm:')
ct = pd.crosstab(obs['population'], obs['arm']).reindex([p for p in C.POP_ORDER if p in set(obs['population'].dropna())])
print(ct.to_string())
print(f'\nunlabelled: {int(obs["population"].isna().sum())}')

population x arm:
arm         cells  nuclei
population               
Fb-interlo  18363    4591
Fb-perilo   23063    5218
Fb-med       7517    1260
Fb-interm   14815    2926
Fb-TGFB+     1284     273
Fb-adipo     5216     590

unlabelled: 0


## 1 · Mean usage share per population and arm

In [2]:
fig, tabs = C.usage_heatmap(U, obs, 'population', arms=C.ARMS, factor_labels=C.FACTOR_LABELS, mode='z',
                            min_cells=MIN_CELLS, width='one-half', cell_mm=5.0,
                            suptitle='iNMF factor usage — z across populations, within arm')
C.save(fig, f'{PLT}/factor_usage_z_by_population.pdf')
plt.show()
for arm, t in tabs.items():
    t.to_csv(f'{RES}/mean_usage_share_{arm}.csv')
    print(f'--- {C.ARM_NAMES[arm]}: mean usage share (populations x factors)')
    print(t.round(3).to_string(), '\n')

-> /nfs/team205/vk8/projects/thymus_ageing_atlas/FB_Vascular_compartment/plots/final/03_usage_by_population/factor_usage_z_by_population.pdf   130 x 107 mm
--- scRNA-seq (cells): mean usage share (populations x factors)
               S1     S2     S3     S4     S5     S6     S7     S8     S9    S10    S11    S12    S13    S14    S15    S16
Fb-interlo  0.032  0.012  0.204  0.008  0.047  0.069  0.052  0.023  0.157  0.083  0.066  0.127  0.023  0.027  0.009  0.061
Fb-perilo   0.013  0.016  0.043  0.008  0.207  0.115  0.044  0.125  0.023  0.052  0.200  0.009  0.031  0.048  0.018  0.048
Fb-med      0.013  0.023  0.019  0.009  0.117  0.138  0.029  0.093  0.010  0.040  0.070  0.006  0.160  0.239  0.008  0.026
Fb-interm   0.040  0.009  0.048  0.005  0.049  0.087  0.386  0.021  0.013  0.024  0.085  0.090  0.015  0.043  0.012  0.071
Fb-TGFB+    0.287  0.010  0.033  0.096  0.004  0.022  0.098  0.006  0.032  0.204  0.028  0.075  0.024  0.019  0.008  0.054
Fb-adipo    0.012  0.008  0.032  0.007  0.

In [3]:
fig, _ = C.usage_heatmap(U, obs, 'population', arms=C.ARMS, factor_labels=C.FACTOR_LABELS, mode='abs',
                         min_cells=MIN_CELLS, width='one-half', cell_mm=5.0,
                         suptitle='iNMF factor usage — mean share (absolute)')
C.save(fig, f'{PLT}/factor_usage_abs_by_population.pdf')
plt.show()

-> /nfs/team205/vk8/projects/thymus_ageing_atlas/FB_Vascular_compartment/plots/final/03_usage_by_population/factor_usage_abs_by_population.pdf   130 x 107 mm


## 2 · How much, and how widely: dotplot

Mean usage alone cannot separate "every cell uses it a little" from "a few cells use it a lot".
Dot colour is the mean share; dot area is the fraction of cells of the population whose share
exceeds 1/k = 0.0625, i.e. more than an even split over the 16 programmes.

In [4]:
THR = 1.0 / C.K
fracs = {}
for arm in C.ARMS:
    m = (obs['arm'] == arm).values & obs['population'].notna().values
    lab = obs.loc[m, 'population'].astype(str)
    f = (U[m] > THR).groupby(lab.values, observed=True).mean()
    fracs[arm] = f.loc[[p for p in tabs[arm].index if p in f.index]]
    fracs[arm].to_csv(f'{RES}/fraction_cells_above_1_over_k_{arm}.csv')

ncol = {a: len(tabs[a].index) for a in C.ARMS}
vmax = max(float(t.values.max()) for t in tabs.values())
fig, axes = plt.subplots(1, 2, figsize=C.figsize_mm('one-half', 95), layout='constrained',
                         gridspec_kw={'width_ratios': [ncol[a] for a in C.ARMS]}, sharey=True)
for ax, arm in zip(axes, C.ARMS):
    t, f = tabs[arm], fracs[arm]
    for i, fac in enumerate(C.FACTORS):
        sc_ = ax.scatter(np.arange(len(t.index)), [i] * len(t.index), s=4 + 90 * f[fac].values,
                         c=t[fac].values, cmap=C.CMAP_USAGE, vmin=0, vmax=vmax, linewidths=0.15, edgecolors='0.35')
    ax.set_xticks(range(len(t.index))); ax.set_xticklabels(list(t.index), rotation=90)
    for i, pop in enumerate(t.index):
        ax.get_xticklabels()[i].set_color(C.POP_COLORS.get(pop, 'k'))
    ax.set_yticks(range(len(C.FACTORS))); ax.set_yticklabels([C.FACTOR_LABELS[f_] for f_ in C.FACTORS])
    ax.set_ylim(len(C.FACTORS) - 0.4, -0.6); ax.set_xlim(-0.6, len(t.index) - 0.4)
    ax.set_title(C.ARM_NAMES[arm]); ax.grid(False)
    ax.tick_params(top=False, right=False)
cb = fig.colorbar(sc_, ax=axes, shrink=0.4, pad=0.02); cb.set_label('mean usage share')
for frac, lab in [(0.1, '10%'), (0.5, '50%'), (1.0, '100%')]:
    axes[-1].scatter([], [], s=4 + 90 * frac, c='0.6', linewidths=0.15, edgecolors='0.35', label=lab)
axes[-1].legend(title='cells above 1/k', loc='upper left', bbox_to_anchor=(1.02, 1.0), frameon=False, labelspacing=1.2, borderaxespad=0)
C.save(fig, f'{PLT}/factor_usage_dotplot_by_population.pdf')
plt.show()

-> /nfs/team205/vk8/projects/thymus_ageing_atlas/FB_Vascular_compartment/plots/final/03_usage_by_population/factor_usage_dotplot_by_population.pdf   130 x 95 mm


## 3 · Cross-modality concordance

For each factor, the Spearman correlation of its mean-usage profile across the populations
present in both arms. This asks whether a programme is used by the *same* populations in
dissociated cells and in nuclei — the property that makes it a shared programme rather than an
assay artefact. Absolute shares are used (the z is a monotone transform within arm, so the rank
correlation is identical).

In [5]:
shared_pops = [p for p in tabs['cells'].index if p in tabs['nuclei'].index]
A, B = tabs['cells'].loc[shared_pops], tabs['nuclei'].loc[shared_pops]
conc = pd.DataFrame({'name': pd.Series(C.FACTOR_NAMES),
                     'rho': [round(float(spearmanr(A[f], B[f]).statistic), 3) for f in C.FACTORS],
                     'top_population_cells': A.idxmax(), 'top_population_nuclei': B.idxmax()}).loc[C.FACTORS]
conc['top_agrees'] = conc['top_population_cells'] == conc['top_population_nuclei']
conc.to_csv(f'{RES}/crossmodality_concordance.csv')
print(f'populations compared: {shared_pops}')
display(conc)
print(f'median rho {conc["rho"].median():.2f}; factors with rho > 0.5: {int((conc["rho"] > 0.5).sum())}/16; '
      f'top population agrees for {int(conc["top_agrees"].sum())}/16')

fig, ax = plt.subplots(figsize=C.figsize_mm('single', 62), layout='constrained')
y = np.arange(len(C.FACTORS))
ax.barh(y, conc['rho'].values, color=[C.COLORS['blue'] if v > 0.5 else C.COLORS['magenta'] for v in conc['rho']], height=0.7)
ax.set_yticks(y); ax.set_yticklabels([C.FACTOR_LABELS[f] for f in C.FACTORS]); ax.invert_yaxis()
ax.axvline(0, color='k', lw=0.5); ax.axvline(0.5, color='0.6', lw=0.5, ls=':')
ax.set_xlabel(f'Spearman ρ of usage across {len(shared_pops)} populations (cells vs nuclei)'); ax.grid(False)
C.save(fig, f'{PLT}/crossmodality_concordance.pdf')
plt.show()

populations compared: ['Fb-interlo', 'Fb-perilo', 'Fb-med', 'Fb-interm', 'Fb-TGFB+', 'Fb-adipo']


,name,rho,top_population_cells,top_population_nuclei,top_agrees
S1,TGFb / hypoxia / mechano,0.943,Fb-TGFB+,Fb-TGFB+,True
S2,lymphoid (contamination),1.000,Fb-med,Fb-med,True
S3,interlobular (CCN5),0.714,Fb-interlo,Fb-interlo,True
S4,proliferation,0.829,Fb-TGFB+,Fb-TGFB+,True
S5,perilobular,0.943,Fb-perilo,Fb-perilo,True
S6,IEG / dissociation,0.886,Fb-med,Fb-med,True
S7,inflammatory,0.886,Fb-interm,Fb-interm,True
S8,perilobular (nuclear),0.943,Fb-perilo,Fb-perilo,True
S9,interlobular (PI16),0.829,Fb-interlo,Fb-interlo,True
S10,ECM synthesis,0.543,Fb-TGFB+,Fb-TGFB+,True


median rho 0.89; factors with rho > 0.5: 14/16; top population agrees for 16/16
-> /nfs/team205/vk8/projects/thymus_ageing_atlas/FB_Vascular_compartment/plots/final/03_usage_by_population/crossmodality_concordance.pdf   89 x 62 mm


## 4 · What each population's usage budget is spent on

In [6]:
cmap = plt.get_cmap('tab20')
fig, axes = plt.subplots(1, 2, figsize=C.figsize_mm('one-half', 60), layout='constrained',
                         gridspec_kw={'width_ratios': [ncol[a] for a in C.ARMS]}, sharey=True)
for ax, arm in zip(axes, C.ARMS):
    t = tabs[arm]
    bottom = np.zeros(len(t))
    for i, f in enumerate(C.FACTORS):
        ax.bar(range(len(t)), t[f].values, bottom=bottom, color=cmap(i % 20), width=0.75, linewidth=0,
               label=C.FACTOR_LABELS[f] if arm == C.ARMS[-1] else None)
        bottom += t[f].values
    ax.set_xticks(range(len(t))); ax.set_xticklabels(list(t.index), rotation=90)
    ax.set_title(C.ARM_NAMES[arm]); ax.set_ylim(0, 1); ax.grid(False)
axes[0].set_ylabel('mean usage share')
axes[-1].legend(bbox_to_anchor=(1.02, 1), loc='upper left', frameon=False, fontsize=4.5, borderaxespad=0)
C.save(fig, f'{PLT}/factor_usage_composition_by_population.pdf')
plt.show()

-> /nfs/team205/vk8/projects/thymus_ageing_atlas/FB_Vascular_compartment/plots/final/03_usage_by_population/factor_usage_composition_by_population.pdf   130 x 60 mm


## Files written

| file | contents |
|---|---|
| `results/final/03_usage_by_population/mean_usage_share_{cells,nuclei}.csv` | populations × factors, mean share |
| `results/final/03_usage_by_population/fraction_cells_above_1_over_k_*.csv` | dot sizes of the dotplot |
| `results/final/03_usage_by_population/crossmodality_concordance.csv` | per-factor Spearman ρ between arms |
| `plots/final/03_usage_by_population/factor_usage_z_by_population.pdf` | **the main figure** — z within arm, cells and nuclei side by side |
| `plots/final/03_usage_by_population/factor_usage_abs_by_population.pdf`, `..._dotplot_...`, `..._composition_...`, `crossmodality_concordance.pdf` | companions |